# SDTM CT Term NCIt Synonyms — NCIt synonyms for every SDTM CT term

Writes `sdtm-test-codes/interim/SDTM_CT_Term_NCIt_Synonyms.csv`: one row per NCIt code of a term
in SDTM CT (every codelist, not only test codes), with the NCIt synonyms from the NCI EVS FLAT file.

The four Findings consumers in `sdtm-findings-graph/` read it for the `term_ncit_synonyms` column
of their `Codelists` sheet. SDTM CT itself carries CDISC synonyms only; NCIt synonyms are NCI's,
and the FLAT file does not say where each synonym comes from.

## Rule

The same rule as `NCIt_Synonyms` in `SDTM_CT_NCIt_Enrich` (SDTM_Test_Identity): the first FLAT
entry is the preferred name and is dropped; the rest are deduplicated ignoring case, in FLAT order,
and joined with `|`, the FLAT file's own separator (as `NCIt_Synonyms` in SDTM_Test_Identity since
2026-10-30). Some NCIt synonyms contain `;` (e.g. *Esophageal; Distal*); `|` never occurs inside one.

## Version

The FLAT file comes from a URL without a version and carries no release number. The pin
`NCIt` in `Reference_Versions.json` is the build date of `Thesaurus.txt` inside
`Thesaurus.FLAT.zip`; the run **fails** when they differ. The date is written to every row
(`NCIt_FLAT_Date`) so the consumers can check it too.

## Inputs

| File | Use |
|---|---|
| `downloads/SDTM_Terminology.txt` | the term codes (rows with a Codelist Code) |
| `downloads/Thesaurus.FLAT.zip`, `downloads/Thesaurus.txt` | NCIt synonyms; the build date |

Both are downloaded by `SDTM_CT_NCIt_Enrich`. Run this notebook after it.

The run **fails** when a term code is not in the FLAT file.


## 1. Setup

In [1]:
import json
import zipfile
from datetime import date
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent          # sdtm-test-codes/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))

SDTM_FILE = BASE_DIR / 'downloads' / 'SDTM_Terminology.txt'
FLAT_ZIP = BASE_DIR / 'downloads' / 'Thesaurus.FLAT.zip'
FLAT_FILE = BASE_DIR / 'downloads' / 'Thesaurus.txt'
OUTPUT_FILE = BASE_DIR / 'interim' / 'SDTM_CT_Term_NCIt_Synonyms.csv'

for f in [SDTM_FILE, FLAT_ZIP, FLAT_FILE]:
    if not f.exists():
        raise FileNotFoundError(f'{f} not found: run SDTM_CT_NCIt_Enrich first')
    print(f'  {f.relative_to(REPO_ROOT)}')


  sdtm-test-codes/downloads/SDTM_Terminology.txt
  sdtm-test-codes/downloads/Thesaurus.FLAT.zip
  sdtm-test-codes/downloads/Thesaurus.txt


## 2. Check the NCIt pin against the FLAT build date

In [2]:
with zipfile.ZipFile(FLAT_ZIP) as zf:
    _info = zf.getinfo('Thesaurus.txt')
FLAT_DATE = date(*_info.date_time[:3]).isoformat()
if _info.file_size != FLAT_FILE.stat().st_size:
    raise RuntimeError(f'{FLAT_FILE.name} ({FLAT_FILE.stat().st_size:,} bytes) is not the file in '
                       f'{FLAT_ZIP.name} ({_info.file_size:,} bytes): unzip again')
if FLAT_DATE != VERSIONS['NCIt']:
    raise RuntimeError(f"NCIt pin {VERSIONS['NCIt']} != FLAT build date {FLAT_DATE}: update Reference_Versions.json")
print(f'NCIt FLAT build date: {FLAT_DATE} (= pin)')


NCIt FLAT build date: 2026-09-29 (= pin)


## 3. Term codes and NCIt synonyms

In [3]:
ct = pd.read_csv(SDTM_FILE, sep='\t', dtype=str).fillna('')
term_codes = sorted(set(ct.loc[ct['Codelist Code'] != '', 'Code']))
print(f'SDTM CT term codes: {len(term_codes):,}')

FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']
flat = pd.read_csv(FLAT_FILE, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
flat = flat[flat['code'].isin(term_codes)]
missing = sorted(set(term_codes) - set(flat['code']))
if missing:
    raise RuntimeError(f'{len(missing)} SDTM CT term codes not in the FLAT file, e.g. {missing[:10]}')


def ncit_synonyms(value):
    entries = [s.strip() for s in value.split('|') if s.strip()][1:]   # drop the preferred name
    seen, kept = set(), []
    for s in entries:
        if s.lower() not in seen:
            seen.add(s.lower())
            kept.append(s)
    return kept


rows = []
semicolon = 0
for code, value in zip(flat['code'], flat['synonyms']):
    syns = ncit_synonyms(value)
    semicolon += sum(';' in s for s in syns)
    rows.append({'NCIt_Code': code, 'NCIt_Synonyms': '|'.join(syns), 'NCIt_FLAT_Date': FLAT_DATE})
out = pd.DataFrame(rows).sort_values('NCIt_Code').reset_index(drop=True)
print(f'Rows: {len(out):,}; with NCIt synonyms: {(out["NCIt_Synonyms"] != "").sum():,}')
print(f'NCIt synonyms that contain ";" (kept whole by the "|" separator): {semicolon}')


SDTM CT term codes: 26,965


Rows: 26,965; with NCIt synonyms: 26,945
NCIt synonyms that contain ";" (kept whole by the "|" separator): 81


## 4. Write

In [4]:
out.to_csv(OUTPUT_FILE, index=False)
print(f'Written: {OUTPUT_FILE.relative_to(REPO_ROOT)}')


Written: sdtm-test-codes/interim/SDTM_CT_Term_NCIt_Synonyms.csv
